In [8]:
# IMPORTS
import os, warnings, csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


In [9]:
# DEVICE
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

Device: cuda


In [ ]:
# CONFIGURATION

# Saved model weights from resnet_full_evaluation.ipynb
MODEL_183 = os.path.expanduser(
    '~/Desktop/results_resnet_full_evaluation/rock_classifier_resnet18_1.83Hz.pth')
MODEL_510 = os.path.expanduser(
    '~/Desktop/results_resnet_full_evaluation/rock_classifier_resnet18_5.10Hz.pth')

# New test data 
TEST_DATA_ROOT = os.path.expanduser('~/for_test_data_spectral_224')

# Results output
RESULTS_DIR = 'results_inference_new_data'
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES  = ['S10Granite', 'Holstein_Sandstone', 'Leitendorf_Limestone']
SHORT_NAMES  = ['Granite', 'Sandstone', 'Limestone']
CLASS_COLORS = ['#4e79a7', '#f28e2b', '#59a14f']

OOD_THRESHOLD = 0.80

# Save helper
_saved_files = []
def save_fig(fig, filename, description, dpi=150):
    path = os.path.join(RESULTS_DIR, filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    _saved_files.append((path, description))
    print(f'[SAVED] {path}')
    print(f'        -> {description}')

for label, p in [('1.83 Hz model', MODEL_183), ('5.10 Hz model', MODEL_510),
                  ('Test data root', TEST_DATA_ROOT)]:
    status = 'OK' if os.path.exists(p) else 'NOT FOUND'
    print(f'  {label:<20} {status}  ({p})')

missing = [p for _, p in [('', MODEL_183), ('', MODEL_510)] if not os.path.exists(p)]
if missing:
    print('\n[ERROR] One or more .pth files not found.')
    print('        Make sure you ran the "SAVE BEST MODELS" cell in resnet_full_evaluation.ipynb.')
    print('        That cell copies the best model to results_resnet_full_evaluation/*.pth')

  1.83 Hz model        NOT FOUND  (/home/puneeth/Desktop/results_resnet_full_evaluation/rock_classifier_resnet18_1.83Hz.pth)
  5.10 Hz model        NOT FOUND  (/home/puneeth/Desktop/results_resnet_full_evaluation/rock_classifier_resnet18_5.10Hz.pth)
  Test data root       OK  (/home/puneeth/for_test_data_spectral_224)

[ERROR] One or more .pth files not found.
        Make sure you ran the "SAVE BEST MODELS" cell in resnet_full_evaluation.ipynb.
        That cell copies the best model to results_resnet_full_evaluation/*.pth


In [ ]:
# TRANSFORMS 
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])
print('Transforms ready.')

Transforms ready.


In [ ]:
# DATASET FOR INFERENCE 
VALID_EXT = ('.jpg', '.jpeg', '.bmp', '.png')

class UnlabelledDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths = paths; self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, os.path.basename(self.paths[idx])

print('UnlabelledDataset ready.')

UnlabelledDataset ready.


In [ ]:
# MODEL BUILDER 
def build_resnet18(n_classes=3):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.fc.in_features, n_classes)
    )
    return model

# LOAD SAVED WEIGHTS
def load_model(pth_path):
    model = build_resnet18(len(CLASS_NAMES)).to(device)
    model.load_state_dict(
        torch.load(pth_path, map_location=device, weights_only=True)
    )
    model.eval()
    return model

model_183 = load_model(MODEL_183)
model_510 = load_model(MODEL_510)
print(f'Loaded 1.83 Hz model  <- {MODEL_183}')
print(f'Loaded 5.10 Hz model  <- {MODEL_510}')

FileNotFoundError: [Errno 2] No such file or directory: '/home/puneeth/Desktop/results_resnet_full_evaluation/rock_classifier_resnet18_1.83Hz.pth'

In [ ]:
# DISCOVER TEST FOLDERS
test_root = Path(TEST_DATA_ROOT)
test_folders = sorted([d for d in test_root.iterdir() if d.is_dir()])

print(f'Found {len(test_folders)} test folders:\n')
print(f'{"Folder":<55} {"Images":>6}  {"Model used"}')
print('-' * 80)
for tf in test_folders:
    n = sum(1 for f in tf.iterdir() if f.suffix.lower() in VALID_EXT)
    speed = '5.10 Hz' if '5-10Hz' in tf.name else '1.83 Hz'
    print(f'{tf.name:<55} {n:>6}  {speed}')

In [ ]:
# RUN INFERENCE
def run_inference(model, image_paths, batch_size=64):
    """Returns preds, confs, probs_all, filenames for a list of image paths."""
    nw      = min(4, os.cpu_count() or 1)
    pin_mem = (device.type == 'cuda')
    ds  = UnlabelledDataset(image_paths, eval_transform)
    ldr = DataLoader(ds, batch_size=batch_size, shuffle=False,
                      num_workers=nw, pin_memory=pin_mem, persistent_workers=(nw > 0))
    preds, confs, probs_all, filenames = [], [], [], []
    with torch.no_grad():
        for Xb, fnames in ldr:
            probs = torch.softmax(model(Xb.to(device)), dim=1).cpu().numpy()
            for prob, fname in zip(probs, fnames):
                preds.append(int(np.argmax(prob)))
                confs.append(float(np.max(prob)))
                probs_all.append(prob.tolist())
                filenames.append(fname)
    return preds, confs, probs_all, filenames


folder_results = {}

for tf in test_folders:
    image_paths = sorted([
        str(f) for f in tf.iterdir() if f.suffix.lower() in VALID_EXT
    ])
    if not image_paths:
        print(f'[SKIP] {tf.name} — no images'); continue

    model_to_use = model_510 if '5-10Hz' in tf.name else model_183
    speed_used   = '5.10 Hz' if '5-10Hz' in tf.name else '1.83 Hz'

    print(f'  {tf.name}  [{speed_used}]  {len(image_paths)} images', end=' ... ', flush=True)
    preds, confs, probs_all, filenames = run_inference(model_to_use, image_paths)

    n_ood        = sum(1 for c in confs if c < OOD_THRESHOLD)
    top_pred_idx = int(np.bincount(preds).argmax())

    folder_results[tf.name] = {
        'preds': preds, 'confs': confs, 'probs_all': probs_all,
        'filenames': filenames, 'n_images': len(image_paths),
        'n_ood': n_ood, 'speed': speed_used,
        'top_pred': CLASS_NAMES[top_pred_idx],
        'mean_conf': float(np.mean(confs)),
    }
    print(f'top pred: {SHORT_NAMES[top_pred_idx]}  '
          f'mean conf: {np.mean(confs)*100:.1f}%  '
          f'uncertain: {n_ood}/{len(image_paths)}')

folder_names = list(folder_results.keys())
n_folders    = len(folder_names)
print(f'\nDone. {n_folders} folders processed.')

In [ ]:
# Short display labels for x-axis
short_labels = [
    fn.replace('Limestone_', 'Lst_')
      .replace('Granite_3SamplesPhilipp', 'Gran_Phil')
      .replace('Sandstone', 'Sst')
      .replace('Dunite-Ecologite_2Rocks', 'Dun-Eco')
      .replace('CalcsilicaContaminated_U9_U3', 'CalcSil')
      .replace('Gneis', 'Gneis')
    for fn in folder_names
]
print('Short labels:', short_labels)

In [ ]:
# INF-01  Prediction distribution bar chart per folder
# What it shows:
#   For each new test folder, the % of images predicted as each of the 3 trained classes.
#   Mean confidence is annotated above each bar.
#   A red warning count (N) = images below the OOD confidence threshold.

fig, ax = plt.subplots(figsize=(max(14, n_folders * 1.7), 6))
fig.suptitle(
    'INF-01 | Prediction Distribution per New Test Folder\n'
    'Each bar = one new rock folder  |  Colour = which trained class was predicted\n'
    f'Number above bar = mean confidence  |  Red ⚠ N = images below {OOD_THRESHOLD*100:.0f}% confidence (uncertain)',
    fontsize=11, fontweight='bold'
)
x = np.arange(n_folders); bottom = np.zeros(n_folders)

for ci, (cls, color, short) in enumerate(zip(CLASS_NAMES, CLASS_COLORS, SHORT_NAMES)):
    fracs = [
        sum(1 for p in folder_results[fn]['preds'] if p == ci) / folder_results[fn]['n_images'] * 100
        for fn in folder_names
    ]
    ax.bar(x, fracs, bottom=bottom, color=color, label=short, width=0.65)
    for xi, (frac, bot) in enumerate(zip(fracs, bottom)):
        if frac > 7:
            ax.text(xi, bot + frac/2, f'{frac:.0f}%',
                    ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')
    bottom += np.array(fracs)

for xi, fn in enumerate(folder_names):
    mc    = folder_results[fn]['mean_conf'] * 100
    n_ood = folder_results[fn]['n_ood']
    ax.text(xi, 102, f'{mc:.0f}%\nconf', ha='center', va='bottom', fontsize=7, color='#333333')
    if n_ood > 0:
        ax.text(xi, 110, f'\u26a0 {n_ood}', ha='center', va='bottom', fontsize=7, color='red')

ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('% of images predicted as class')
ax.set_ylim(0, 118)
ax.set_yticks(range(0, 101, 20))
ax.legend(title='Predicted class', loc='upper right', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
ax.axhline(100, color='black', lw=0.5)

plt.tight_layout()
save_fig(fig, 'INF-01_prediction_distribution__per_test_folder.png',
    'Stacked bar chart: % of images per new folder predicted as each trained class, '
    'with mean confidence and uncertain-image count annotated above each bar')
plt.show()

In [ ]:
# INF-02  Confidence and uncertainty heatmap
# What it shows:
#   Two-column heatmap per folder.
#   Left column  = mean confidence (max softmax probability averaged over all images in the folder).
#   Right column = % of images flagged as uncertain (confidence below OOD_THRESHOLD).

mean_confs  = [folder_results[fn]['mean_conf'] * 100      for fn in folder_names]
pct_uncerts = [folder_results[fn]['n_ood'] / folder_results[fn]['n_images'] * 100
               for fn in folder_names]

mat = np.array([mean_confs, pct_uncerts]).T

fig, ax = plt.subplots(figsize=(5, max(5, n_folders * 0.6)))
fig.suptitle(
    f'INF-02 | Confidence & Uncertainty per New Test Folder\n'
    f'Left = mean max-softmax confidence (%)  |  Right = % images below {OOD_THRESHOLD*100:.0f}% threshold\n'
    f'High right-column value = model does not recognise this rock (out-of-distribution)',
    fontsize=10, fontweight='bold'
)
sns.heatmap(
    mat, ax=ax,
    xticklabels=['Mean Confidence (%)', f'Uncertain (< {OOD_THRESHOLD*100:.0f}%) (%)'],
    yticklabels=short_labels,
    annot=True, fmt='.1f', cmap='RdYlGn', vmin=0, vmax=100,
    linewidths=0.5, linecolor='white', cbar_kws={'label': '%'}
)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
save_fig(fig, 'INF-02_confidence_and_uncertainty_heatmap__per_test_folder.png',
    f'Heatmap: mean confidence and % uncertain images (below {OOD_THRESHOLD*100:.0f}% threshold) per new test folder')
plt.show()

In [ ]:
# INF-03  Confidence distribution histograms per folder
# What it shows:
#   One histogram per folder showing the distribution of max-softmax confidence scores.
#   A sharp spike near 1.0 = model is consistently very confident about that rock.
#   A spread distribution or cluster near 0.33 = model is uncertain / out-of-distribution.
#   The bar colour = top predicted class for that folder.
#   Red dashed line = OOD threshold.

ncols = 3
nrows = int(np.ceil(n_folders / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = axes.flatten()
fig.suptitle(
    'INF-03 | Confidence Distribution Histograms per New Test Folder\n'
    'X-axis = max softmax probability per image  |  Y-axis = number of images\n'
    'Spike near 1.0 = confident  |  Spread / spike near 0.33 = uncertain or out-of-distribution\n'
    f'Red dashed line = OOD threshold ({OOD_THRESHOLD*100:.0f}%)',
    fontsize=11, fontweight='bold'
)
for ax, fn in zip(axes, folder_names):
    confs  = folder_results[fn]['confs']
    n_ood  = folder_results[fn]['n_ood']
    color  = CLASS_COLORS[CLASS_NAMES.index(folder_results[fn]['top_pred'])]
    ax.hist(confs, bins=25, range=(0, 1), color=color, alpha=0.8, edgecolor='white', lw=0.3)
    ax.axvline(OOD_THRESHOLD, color='red', ls='--', lw=1.2)
    lbl = short_labels[folder_names.index(fn)]
    ax.set_title(
        f'{lbl}\n'
        f'top: {folder_results[fn]["top_pred"].split("_")[0]}  '
        f'mean conf: {np.mean(confs)*100:.1f}%  '
        f'uncertain: {n_ood}/{len(confs)}',
        fontsize=8
    )
    ax.set_xlabel('Confidence', fontsize=8)
    ax.set_ylabel('# images', fontsize=8)
    ax.set_xlim(0, 1); ax.grid(True, alpha=0.3)
for ax in axes[n_folders:]: ax.set_visible(False)

plt.tight_layout()
save_fig(fig, 'INF-03_confidence_histograms__per_test_folder.png',
    'Per-folder confidence histograms: reveals whether the model treats each new rock '
    'as in-distribution (spike near 1.0) or OOD (spread or cluster near 0.33)')
plt.show()

In [ ]:
# INF-04  Mean softmax probability vector per folder
# What it shows:
#   For each folder, the average probability assigned to each of the 3 trained classes
#   averaged over all images in that folder.
#   A bar dominated by one colour = the model consistently assigns this rock to that class.
#   An even split between colours = the model is hedging.

mean_probs = np.array([
    np.mean(folder_results[fn]['probs_all'], axis=0)
    for fn in folder_names
])

fig, ax = plt.subplots(figsize=(max(12, n_folders * 1.7), 6))
fig.suptitle(
    'INF-04 | Mean Softmax Probability per Class per New Test Folder\n'
    'Each bar = one folder  |  Colour = trained class  |  Values = mean probability across all images\n'
    'One dominant colour = model assigns most probability to that class\n'
    'Even colour split = model is uncertain (typical for rocks not seen during training)',
    fontsize=11, fontweight='bold'
)
x = np.arange(n_folders); bottom = np.zeros(n_folders)
for ci, (cls, color, short) in enumerate(zip(CLASS_NAMES, CLASS_COLORS, SHORT_NAMES)):
    vals = mean_probs[:, ci] * 100
    ax.bar(x, vals, bottom=bottom, color=color, label=short, width=0.65)
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 5:
            ax.text(xi, b + v/2, f'{v:.0f}%',
                    ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Mean softmax probability (%)')
ax.set_ylim(0, 107)
ax.set_yticks(range(0, 101, 20))
ax.legend(title='Trained class', loc='upper right', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
ax.axhline(100, color='black', lw=0.5)

plt.tight_layout()
save_fig(fig, 'INF-04_mean_softmax_probabilities__per_test_folder.png',
    'Mean softmax probability per class per folder: shows how probability mass is '
    'distributed across the 3 trained classes for each new rock type')
plt.show()

In [ ]:
# INF-05  Summary table
print(f'\n{"="*115}')
print('INFERENCE SUMMARY — New Test Data')
print(f'Model source: results_resnet_full_evaluation/  |  OOD threshold: {OOD_THRESHOLD*100:.0f}%')
print(f'{"="*115}')
print(f'{"Folder":<55} {"Model":>7} {"N":>5} {"Top pred":<12} '
      f'{"Mean conf":>9} {"Uncert":>9}  {"P(Gran)":>7} {"P(Sst)":>7} {"P(Lst)":>7}')
print('-' * 115)
for fn in folder_names:
    r  = folder_results[fn]
    mp = np.mean(r['probs_all'], axis=0)
    print(
        f'{fn:<55} {r["speed"]:>7} {r["n_images"]:>5} '
        f'{r["top_pred"].split("_")[0]:<12} '
        f'{r["mean_conf"]*100:>8.1f}% '
        f'{r["n_ood"]:>5}/{r["n_images"]:<5} '
        f'{mp[0]*100:>7.1f}% {mp[1]*100:>6.1f}% {mp[2]*100:>6.1f}%'
    )
print('=' * 115)

In [ ]:
# SAVE CSVs

# Per-image predictions
csv_images = os.path.join(RESULTS_DIR, 'predictions_all_images.csv')
with open(csv_images, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['folder', 'belt_speed', 'filename', 'predicted_class', 'confidence',
                'uncertain', 'prob_S10Granite', 'prob_Holstein_Sandstone', 'prob_Leitendorf_Limestone'])
    for fn in folder_names:
        r = folder_results[fn]
        for fname, pred, conf, probs in zip(r['filenames'], r['preds'], r['confs'], r['probs_all']):
            w.writerow([fn, r['speed'], fname, CLASS_NAMES[pred],
                        round(conf, 4), 'YES' if conf < OOD_THRESHOLD else 'NO',
                        round(probs[0], 4), round(probs[1], 4), round(probs[2], 4)])
_saved_files.append((csv_images, 'Per-image: folder, speed, filename, predicted class, confidence, OOD flag, full probability vector'))
print(f'[SAVED] {csv_images}')

# Per-folder summary
csv_folders = os.path.join(RESULTS_DIR, 'predictions_per_folder.csv')
with open(csv_folders, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['folder', 'belt_speed', 'n_images', 'top_predicted_class',
                'mean_confidence_pct', 'n_uncertain', 'pct_uncertain',
                'pct_pred_Granite', 'pct_pred_Sandstone', 'pct_pred_Limestone',
                'mean_prob_Granite', 'mean_prob_Sandstone', 'mean_prob_Limestone'])
    for fn in folder_names:
        r = folder_results[fn]
        mp = np.mean(r['probs_all'], axis=0)
        counts = np.bincount(r['preds'], minlength=3)
        w.writerow([fn, r['speed'], r['n_images'], r['top_pred'],
                    round(r['mean_conf']*100, 2), r['n_ood'],
                    round(r['n_ood']/r['n_images']*100, 2),
                    round(counts[0]/r['n_images']*100, 2),
                    round(counts[1]/r['n_images']*100, 2),
                    round(counts[2]/r['n_images']*100, 2),
                    round(mp[0]*100, 2), round(mp[1]*100, 2), round(mp[2]*100, 2)])
_saved_files.append((csv_folders, 'Per-folder: top class, mean confidence, uncertain count, prediction distribution, mean probabilities'))
print(f'[SAVED] {csv_folders}')

In [ ]:
# RESULTS INDEX
index_path = os.path.join(RESULTS_DIR, 'RESULTS_INDEX.txt')
with open(index_path, 'w') as f:
    f.write('RESULTS INDEX — inference_new_data\n')
    f.write('=' * 80 + '\n')
    f.write(f'Model source : results_resnet_full_evaluation/\n')
    f.write(f'1.83 Hz model: {MODEL_183}\n')
    f.write(f'5.10 Hz model: {MODEL_510}\n')
    f.write(f'Trained classes: {CLASS_NAMES}\n')
    f.write(f'OOD threshold  : {OOD_THRESHOLD*100:.0f}%\n')
    f.write(f'Test folders   : {n_folders}\n')
    f.write(f'Total images   : {sum(r["n_images"] for r in folder_results.values())}\n')
    f.write('=' * 80 + '\n\n')
    f.write('PLOTS\n' + '-'*80 + '\n')
    for path, desc in _saved_files:
        if path.endswith('.png'):
            f.write(f'  {os.path.basename(path)}\n    WHAT IT SHOWS: {desc}\n\n')
    f.write('\nCSVs\n' + '-'*80 + '\n')
    for path, desc in _saved_files:
        if path.endswith('.csv'):
            f.write(f'  {os.path.basename(path)}\n    WHAT IT SHOWS: {desc}\n\n')

print(f'[SAVED] {index_path}')
print(f'\nTotal files saved: {len(_saved_files)}')
with open(index_path) as f:
    print('\n' + f.read())